In [1]:
import pandas as pd
from datetime import date, timedelta, datetime
import os, shutil, pathlib, glob, time, contextlib, io
from nbconvert import PythonExporter
import polars as pl
from IPython.display import display

def convert_notebook(nb_path: str) -> str:
    exporter = PythonExporter()
    source, _ = exporter.from_filename(nb_path)
    return source


In [2]:
RUN_NOTEBOOKS = True

notebook_dict = {
    'IEX_optimized.ipynb'    : 1,
    'RTA_optimized (1).ipynb': 1,
}

PRODUCTIVE_THRESHOLD  = 0.90
TOTAL_LOGGED_THRESHOLD = 90.0

DATE_FROM = None
DATE_TO   = None

LOB_EXCLUDE_CONTAINS   = ['Support', 'Training']
SHIFT_TRACKING_EXCLUDE = [
    'Training Offline', 'Off Phone Misc', 'PO', 'Off', 'Offline',
    'SL', 'AL', 'CO', 'LWP', 'Termination', 'NCNS',
]


In [3]:
_now = datetime.now()
days_back = 2 if _now.hour < 6 else 1

if DATE_TO is None:
    _auto_to = _now.date() - timedelta(days=days_back)
else:
    _auto_to = datetime.strptime(DATE_TO, "%Y-%m-%d").date()

if DATE_FROM is None:
    _auto_from = _auto_to.replace(day=1)
else:
    _auto_from = datetime.strptime(DATE_FROM, "%Y-%m-%d").date()

start_date = str(_auto_from)
end_date   = str(_auto_to)

print(f"{_now.strftime('%Y-%m-%d %H:%M')} | days_back={days_back} | {start_date} → {end_date}")

first_glob = os.path.expanduser("~").replace("\\", "/")
if not os.path.exists(f"{first_glob}/Concentrix Corporation"):
    raise FileNotFoundError(f"Root directory not found")

CODE_PATH = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE"
RAW_PATH  = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata"

folder_paths = {
    "output_iex_base": f"{RAW_PATH}/OUTPUT_AGENT_IEX_BASE",
    "storage_rta"    : f"{RAW_PATH}/STORAGE_OUTPUT_RTA",
    "python_code"    : f"{CODE_PATH}/Python_Code",
    "resources"      : f"{CODE_PATH}/Resources",
}

def convert_to_datetime(struct_time):
    return datetime(*struct_time[:6])

def input_data_glob(data_dir):
    def read_file(filename):
        try:
            mtime = convert_to_datetime(time.localtime(os.path.getmtime(filename)))
            df = (pl.read_excel(filename, infer_schema_length=0)
                  if filename.suffix.lower() == ".xlsx"
                  else pl.read_csv(filename, infer_schema_length=0, encoding="utf-8"))
            return df.with_columns(pl.lit(filename.stem).alias("sheet_name"),
                                   pl.lit(mtime).alias("Export time"))
        except Exception:
            try:
                return (pl.read_csv(filename, infer_schema_length=0,
                                    encoding="ISO-8859-1", ignore_errors=True)
                        .with_columns(pl.lit(filename.stem).alias("sheet_name"),
                                      pl.lit(mtime).alias("Export time")))
            except Exception as e:
                print(f"  Skip {filename.name}: {e}")
                return None
    dfs = [df for f in pathlib.Path(data_dir).glob("**/*.*")
           if f.suffix.lower() in (".xlsx", ".csv")
           and (df := read_file(f)) is not None]
    if not dfs:
        return pl.DataFrame()
    col_dtypes = {col: {df[col].dtype for df in dfs if col in df.columns}
                  for col in set(c for df in dfs for c in df.columns)}
    dfs = [df.with_columns(pl.col(c).cast(pl.Utf8)
                           for c, t in col_dtypes.items()
                           if c in df.columns and len(t) > 1)
           for df in dfs]
    return pl.concat(dfs, how="vertical")

def input_data(folder_path, sheet_name=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []
    for file in file_paths:
        try:
            if file.endswith(".xlsx"):
                df = pl.read_excel(file, sheet_name=sheet_name, infer_schema_length=0)
            else:
                try:
                    df = pl.read_csv(file, encoding="utf-8", infer_schema_length=0)
                except Exception:
                    df = pl.read_csv(file, encoding="ISO-8859-1",
                                     ignore_errors=True, infer_schema_length=0)
            df_list.append(df.with_columns(pl.col(c).cast(pl.String) for c in df.columns))
        except Exception as e:
            print(f"  Skip {os.path.basename(file)}: {e}")
    return pl.concat(df_list, how="vertical") if df_list else pl.DataFrame()

IEX_BASE = input_data_glob(folder_paths["output_iex_base"])
print(f"IEX_BASE loaded: {IEX_BASE.height:,} rows")

if not RUN_NOTEBOOKS:
    print("[SKIP ALL] RUN_NOTEBOOKS = False")
else:
    _nb_dir = folder_paths['python_code']
    for nbook, flag in notebook_dict.items():
        if flag == 0:
            print(f"[SKIP] {nbook}")
            continue
        nbook_path = f"{_nb_dir}/{nbook}"
        if os.path.isfile(nbook_path):
            print(f"[RUN]  {nbook}")
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    exec(convert_notebook(nbook_path), {**globals()})
                print(f"[DONE] {nbook}")
            except Exception as e:
                print(f"[ERROR] {nbook}: {e}")
        else:
            print(f"[NOT FOUND] {nbook_path}")


2026-09-07 03:03 | days_back=2 | 2026-09-01 → 2026-09-05
IEX_BASE loaded: 109,163 rows
[RUN]  IEX_optimized.ipynb


<string>:257: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
<string>:291: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:342: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<string>:587: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
<string>:790: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass 

,Date,IEX_ID,Scheduled Activity,Datetime_Start_Action,Datetime_End_Action,First Shift,Open Time,Extra Time,NCNS,Target,Time_Of_Day,Shift Tracking


,Date,IEX_ID,Datetime_First_Start_Shift,Datetime_First_End_Shift,First Shift,Open Time,Extra Time,Target,Time_Of_Day,Shift Tracking


[DONE] IEX_optimized.ipynb
[RUN]  RTA_optimized (1).ipynb


Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
<string>:379: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`


[DONE] RTA_optimized (1).ipynb


In [4]:
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(100)

RTA_REPORT_GENERATE = input_data(folder_paths["storage_rta"])

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col("Date_Converted").str.slice(0, 10).str.strptime(pl.Date, format="%Y-%m-%d"),
    pl.col("Target").cast(pl.Float64),
    pl.col("duration").cast(pl.Float64),
    pl.col("sum_productive").cast(pl.Float64),
    pl.col("start").str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False),
    pl.col("end").str.strip_chars().str.to_datetime("%Y-%m-%d %H:%M:%S%.3f", strict=False),
    pl.col("IEX ID").cast(pl.Utf8).str.strip_chars().str.strip_chars('"').alias("IEX ID"),
])

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.filter(
    pl.col("duration").is_not_null() | (pl.col("Target") > 0)
)

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.col("Shift Tracking").cast(pl.Utf8).str.strip_chars(),
    pl.col("Open Time").cast(pl.Float64).fill_null(0.0),
    pl.col("Extra Time").cast(pl.Float64).fill_null(0.0),
])

RTA_REPORT_GENERATE = RTA_REPORT_GENERATE.with_columns([
    pl.when(pl.col("Shift Tracking") == "HDL").then(0.5)
    .when(
        pl.col("Shift Tracking").is_in(["PR", "PR - OT", "PO"]) &
        ((pl.col("Open Time") > 0) | (pl.col("Extra Time") > 0))
    ).then(1.0)
    .when(
        pl.col("Shift Tracking").is_in([
            "Training Offline", "Sick Leave", "Paid Leave", "Billable Training"
        ]) & (pl.col("Open Time") == 0)
    ).then(1.0)
    .when(
        ~pl.col("Shift Tracking").is_in(["HDL", "PR"]) &
        (pl.col("Open Time") == 0)
    ).then(0.0)
    .otherwise(None)
    .alias("hc_present")
])

def filter_shift_discrepancies(df: pl.DataFrame) -> pl.DataFrame:
    result = df.filter(
        (
            (pl.col("hc_actual").cast(pl.Float64, strict=False).fill_null(0.0) !=
             pl.col("hc_present").cast(pl.Float64, strict=False).fill_null(0.0)) |
            (
                (pl.col("Target").cast(pl.Float64, strict=False).fill_null(0.0) > 3.75) &
                (pl.col("sum_productive").cast(pl.Float64, strict=False).fill_null(0.0) <= 3.0)
            )
        ) &
        (
            ~pl.col("LOB").str.contains("Support", literal=True) &
            ~pl.col("LOB").str.contains("Training", literal=True)
        ) &
        (
            ~pl.col("First Shift").is_in(["Training", "Offline", "Off"]) &
            ~pl.col("Shift Tracking").is_in(["Training Offline", "Off Phone Misc", "PO"]) &
            (
                pl.col("Shift Tracking").is_in(["PR", "PR - OT"]) |
                (pl.col("sum_productive").cast(pl.Float64, strict=False).fill_null(0.0) > 0.0)
            )
        )
    )
    if "Date_Converted" in result.columns:
        result = result.sort("Date_Converted")
    return result

def _add_hdl_shift(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .with_columns([
            pl.col("First Shift").str.extract(r"^(\d{4})-\d{4}$", 1).alias("_s_start"),
            pl.col("First Shift").str.extract(r"^\d{4}-(\d{4})$", 1).alias("_s_end"),
        ])
        .with_columns([
            (pl.col("_s_start").str.slice(0,2).cast(pl.Int64, strict=False) * 60 +
             pl.col("_s_start").str.slice(2,2).cast(pl.Int64, strict=False)).alias("_s_start_min"),
            (pl.col("_s_end").str.slice(0,2).cast(pl.Int64, strict=False) * 60 +
             pl.col("_s_end").str.slice(2,2).cast(pl.Int64, strict=False)).alias("_s_end_min"),
            (pl.col("start").dt.hour().cast(pl.Int64) * 60 +
             pl.col("start").dt.minute().cast(pl.Int64)).alias("_a_start_min"),
        ])
        .with_columns([
            pl.when(pl.col("_s_end_min") < pl.col("_s_start_min"))
              .then(pl.col("_s_end_min") + 1440 - pl.col("_s_start_min"))
              .otherwise(pl.col("_s_end_min") - pl.col("_s_start_min"))
              .alias("_s_duration_min"),
            (pl.col("_a_start_min") - pl.col("_s_start_min")).alias("_diff"),
        ])
        .with_columns([
            pl.when(pl.col("_diff") < 0)
              .then(pl.col("_diff") + 1440)
              .otherwise(pl.col("_diff"))
              .alias("_a_rel_start"),
        ])
        .with_columns([
            pl.when(pl.col("_s_start").is_null() | pl.col("start").is_null())
              .then(pl.lit(None))
              .when(pl.col("_a_rel_start").cast(pl.Float64) <
                    pl.col("_s_duration_min").cast(pl.Float64) / 2.0)
              .then(pl.lit("First"))
              .otherwise(pl.lit("Second"))
              .alias("HDL_Shift")
        ])
        .drop(["_s_start", "_s_end", "_s_start_min", "_s_end_min",
               "_s_duration_min", "_diff", "_a_rel_start", "_a_start_min"])
    )

_base_cols = [
    "Date_Converted", "Employee Name", "LOB", "Detail Status", "IEX ID",
    "First Shift", "Shift Tracking", "start", "end", "Target", "duration",
    "sum_productive", "training-idle", "Extra Time",
    "hc_schedule", "hc_actual", "hc_present",
]
_avail_base = [c for c in _base_cols if c in RTA_REPORT_GENERATE.columns]

filtered_data = (
    RTA_REPORT_GENERATE
    .select(_avail_base)
    .with_columns([
        pl.col("hc_actual").cast(pl.Float64),
        pl.col("hc_present").cast(pl.Float64),
    ])
    .sort("Date_Converted")
    .filter(
        (pl.col("Date_Converted") >= pl.lit(start_date).str.strptime(pl.Date, "%Y-%m-%d")) &
        (pl.col("Date_Converted") <= pl.lit(end_date).str.strptime(pl.Date, "%Y-%m-%d"))
    )
)

filtered_data_1 = _add_hdl_shift(filter_shift_discrepancies(filtered_data))

input_iex_atd = filtered_data_1.select(["Date_Converted", "IEX ID", "HDL_Shift"])

ncns_hdl_data = (
    RTA_REPORT_GENERATE
    .select(_avail_base)
    .with_columns([
        pl.col("hc_actual").cast(pl.Float64),
        pl.col("hc_present").cast(pl.Float64),
    ])
    .filter(
        (pl.col("Date_Converted") >= pl.lit(start_date).str.strptime(pl.Date, "%Y-%m-%d")) &
        (pl.col("Date_Converted") <= pl.lit(end_date).str.strptime(pl.Date, "%Y-%m-%d")) &
        pl.col("Shift Tracking").is_in(["NCNS", "HDL"])
    )
    .sort("Date_Converted")
)
ncns_hdl_data = _add_hdl_shift(ncns_hdl_data)

IEX_BASE_filtered = IEX_BASE.with_columns(
    pl.col("Agent").str.extract(r"Agent:\s*(\d+)", 1).alias("IEX_ID"),
    pl.col("Date").cast(pl.Utf8)
)
_filter_keys = (
    filtered_data_1
    .select([
        pl.col("IEX ID").cast(pl.Utf8).alias("IEX_ID"),
        pl.col("Date_Converted").cast(pl.Utf8).alias("Date"),
    ])
    .unique()
)
IEX_BASE_filtered = (
    IEX_BASE_filtered
    .join(_filter_keys, on=["IEX_ID", "Date"], how="semi")
    .with_columns(
        pl.col("Date").str.strptime(pl.Date, format="%Y-%m-%d", strict=False),
        pl.col("Generate Date").str.strptime(pl.Datetime, format="%m/%d/%y %I:%M %p", strict=False),
        pl.col("Start_Shift").str.strptime(pl.Time, format="%I:%M %p", strict=False),
        pl.col("End_Shift").str.strptime(pl.Time, format="%I:%M %p", strict=False),
        pl.lit(None).cast(pl.Utf8).alias("Scheduled Activity"),
        pl.lit(None).cast(pl.Utf8).alias("Start_Action"),
        pl.lit(None).cast(pl.Utf8).alias("End_Action"),
        pl.col("sheet_name").str.strip_chars('"').str.replace(r"IEX_Base_", "").alias("sheet_name"),
    )
    .drop("IEX_ID")
    .unique()
    .sort("Date")
)

FLOAT_COLS = [
    "Target", "sum_productive", "duration", "hc_actual", "hc_schedule",
    "Open Time", "Extra Time", "Break Time", "Lunch Time", "Training",
    "Time_Of_Day", "NCNS", "AL",
    "break", "lunch", "over_break", "over_lunch", "exceed_break",
    "time_late", "time_leave", "adherence_time", "lateness",
    "total_time_chat_handle", "other_status", "training-idle",
    "coaching-idle", "outbound-idle", "break_count",
    "OT PreShift", "OT PostShift",
]
_avail_float = [c for c in FLOAT_COLS if c in RTA_REPORT_GENERATE.columns]

RTA_CAST = RTA_REPORT_GENERATE.with_columns([
    pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in _avail_float
])

for _col in ["Datetime_Fluctuate_Start_Shift", "Datetime_Fluctuate_End_Shift",
             "Datetime_First_Start_Shift", "Datetime_First_End_Shift"]:
    if _col in RTA_CAST.columns:
        RTA_CAST = RTA_CAST.with_columns(
            pl.col(_col).str.strip_chars()
              .str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
              .alias(_col)
        )

RTA_CAST = RTA_CAST.with_columns([
    pl.when(pl.col("Target").is_not_null() & (pl.col("Target") > 0))
      .then((pl.col("sum_productive") / pl.col("Target") * 100).round(1))
      .otherwise(None)
      .alias("productive_pct"),

    pl.when(pl.col("start").is_not_null() & pl.col("end").is_not_null())
      .then(((pl.col("end") - pl.col("start")).dt.total_seconds() / 3600.0).round(2))
      .otherwise(None)
      .alias("actual_span_h"),

    pl.max_horizontal(
        pl.lit(0.0),
        (pl.col("Target") - pl.col("sum_productive")).round(2)
    ).alias("productive_gap_h"),

    pl.when(
        pl.col("Datetime_Fluctuate_Start_Shift").is_not_null() & pl.col("start").is_not_null()
    ).then(
        pl.when(
            ((pl.col("start") - pl.col("Datetime_Fluctuate_Start_Shift"))
             .dt.total_seconds() / 60.0).abs() > 720
        )
        .then(None)
        .otherwise(
            ((pl.col("start") - pl.col("Datetime_Fluctuate_Start_Shift"))
             .dt.total_seconds() / 60.0).round(1)
        )
    ).otherwise(None).alias("late_vs_sched_min"),

    pl.when(
        pl.col("Datetime_Fluctuate_End_Shift").is_not_null() & pl.col("end").is_not_null()
    ).then(
        pl.when(
            ((pl.col("Datetime_Fluctuate_End_Shift") - pl.col("end"))
             .dt.total_seconds() / 60.0).abs() > 720
        )
        .then(None)
        .otherwise(
            ((pl.col("Datetime_Fluctuate_End_Shift") - pl.col("end"))
             .dt.total_seconds() / 60.0).round(1)
        )
    ).otherwise(None).alias("early_leave_min"),
])

low_prod = RTA_CAST.filter(
    pl.col("Target").is_not_null() &
    (pl.col("Target") > 0) &
    pl.col("sum_productive").is_not_null() &
    (pl.col("sum_productive") < pl.col("Target") * PRODUCTIVE_THRESHOLD)
).filter(
    (pl.col("Date_Converted") >= pl.lit(start_date).str.strptime(pl.Date, "%Y-%m-%d")) &
    (pl.col("Date_Converted") <= pl.lit(end_date).str.strptime(pl.Date, "%Y-%m-%d"))
)

if "LOB" in low_prod.columns:
    for _kw in LOB_EXCLUDE_CONTAINS:
        low_prod = low_prod.filter(
            pl.col("LOB").is_null() | ~pl.col("LOB").str.contains(_kw, literal=True)
        )

if "Shift Tracking" in low_prod.columns:
    low_prod = low_prod.filter(
        pl.col("Shift Tracking").is_null() |
        ~pl.col("Shift Tracking").is_in(SHIFT_TRACKING_EXCLUDE)
    )

if "start" in low_prod.columns:
    low_prod = low_prod.filter(pl.col("start").is_not_null())

low_prod = low_prod.sort(["Date_Converted", "productive_pct"])

DISPLAY_COLS = [
    "Date_Converted", "Week_Monday",
    "Employee Name", "Email Id", "IEX ID", "OracleID",
    "LOB", "LOB_2", "Detail Status", "Alias", "Supervisor Name",
    "First Shift", "Fluctuate Shift", "Shift Tracking",
    "Datetime_Fluctuate_Start_Shift", "Datetime_Fluctuate_End_Shift",
    "Target",
    "start", "end", "actual_span_h",
    "late_vs_sched_min", "early_leave_min",
    "sum_productive", "productive_pct", "productive_gap_h",
    "duration",
    "break", "over_break", "exceed_break", "break_count",
    "lunch", "over_lunch",
    "other_status", "training-idle", "coaching-idle",
]
_avail_disp = [c for c in DISPLAY_COLS if c in low_prod.columns]

_idle_cols = [c for c in ["training-idle", "coaching-idle", "other_status"] if c in low_prod.columns]
low_prod_display = (
    low_prod
    .with_columns(
        sum(pl.col(c).cast(pl.Float64, strict=False).fill_null(0.0) for c in _idle_cols)
          .alias("_idle_sum")
        if len(_idle_cols) > 1
        else pl.col(_idle_cols[0]).cast(pl.Float64, strict=False).fill_null(0.0).alias("_idle_sum")
    )
    .with_columns([
        (pl.col("sum_productive").fill_null(0.0) + pl.col("_idle_sum"))
          .round(2)
          .alias("total_logged_h"),
        pl.when(pl.col("Target").is_not_null() & (pl.col("Target") > 0))
          .then(
              ((pl.col("sum_productive").fill_null(0.0) + pl.col("_idle_sum"))
               / pl.col("Target") * 100).round(1)
          )
          .otherwise(None)
          .alias("total_logged_pct"),
    ])
    .drop("_idle_sum")
    .select(_avail_disp + [c for c in ["total_logged_h", "total_logged_pct"] if c not in _avail_disp])
)

def _hours_to_hhmm(col: str) -> pl.Expr:
    _v = pl.col(col).fill_null(0.0)
    _total_min = (_v * 60).cast(pl.Int64)
    _h = (_total_min // 60).abs()
    _m = _total_min.abs() % 60
    return (
        pl.when(_v < 0)
          .then(pl.lit("-") + _h.cast(pl.Utf8) + pl.lit(":") + _m.cast(pl.Utf8).str.zfill(2))
          .otherwise(_h.cast(pl.Utf8) + pl.lit(":") + _m.cast(pl.Utf8).str.zfill(2))
    ).alias(col)

def _late_to_hhmm(col: str) -> pl.Expr:
    _v = pl.col(col).fill_null(0.0)
    _total_min = pl.when(_v > 0).then(_v).otherwise(0.0).cast(pl.Int64)
    _h = _total_min // 60
    _m = _total_min % 60
    return (_h.cast(pl.Utf8) + pl.lit(":") + _m.cast(pl.Utf8).str.zfill(2)).alias(col)

def _early_to_hhmm(col: str) -> pl.Expr:
    _v = pl.col(col).fill_null(0.0)
    _total_min = pl.when(_v > 0).then(_v).otherwise(0.0).cast(pl.Int64)
    _h = _total_min // 60
    _m = _total_min % 60
    return (_h.cast(pl.Utf8) + pl.lit(":") + _m.cast(pl.Utf8).str.zfill(2)).alias(col)

def _sec_to_hhmm(col: str) -> pl.Expr:
    return _hours_to_hhmm(col)

HHMM_COLS_HOURS = [
    "actual_span_h", "sum_productive", "productive_gap_h", "duration",
    "break", "over_break", "lunch", "over_lunch",
    "other_status", "training-idle", "coaching-idle", "total_logged_h",
]
HHMM_COLS_MINS = ["late_vs_sched_min", "early_leave_min"]
HHMM_COLS = HHMM_COLS_HOURS + HHMM_COLS_MINS

_report_days = 30
_csv_name    = f"rta_report_{_report_days}_days.csv"
_out         = f"{folder_paths['resources']}/{_csv_name}"

_DT_COLS = [
    "Datetime_Fluctuate_Start_Shift", "Datetime_Fluctuate_End_Shift",
    "Datetime_First_Start_Shift", "Datetime_First_End_Shift",
    "start", "end",
]

_DECIMAL_COLS = [
    "Target", "hc_actual", "hc_schedule", "hc_present",
    "productive_pct", "productive_gap_h", "actual_span_h",
    "late_vs_sched_min", "early_leave_min",
    "total_logged_h", "total_logged_pct",
    "break_count", "exceed_break",
]

_CSV_DISPLAY_COLS = _avail_disp + [
    c for c in ["total_logged_h", "total_logged_pct", "note"]
    if c not in _avail_disp
]

_low_note_ids = (
    low_prod_display
    .filter(
        pl.col("total_logged_pct").is_not_null() &
        (pl.col("total_logged_pct") < TOTAL_LOGGED_THRESHOLD)
    )
    .select(["Date_Converted", "Email Id"])
    .with_columns(pl.lit("Not Enough Productive").alias("_note_flag"))
)

def _format_csv(df: pl.DataFrame) -> pl.DataFrame:
    _idle_cols_csv = [c for c in ["training-idle", "coaching-idle", "other_status"] if c in df.columns]
    if len(_idle_cols_csv) > 1:
        df = df.with_columns(
            sum(pl.col(c).cast(pl.Float64, strict=False).fill_null(0.0) for c in _idle_cols_csv)
              .alias("_idle_sum")
        )
    elif len(_idle_cols_csv) == 1:
        df = df.with_columns(
            pl.col(_idle_cols_csv[0]).cast(pl.Float64, strict=False).fill_null(0.0).alias("_idle_sum")
        )
    else:
        df = df.with_columns(pl.lit(0.0).alias("_idle_sum"))

    df = df.with_columns([
        (pl.col("sum_productive").cast(pl.Float64, strict=False).fill_null(0.0) + pl.col("_idle_sum"))
          .round(2).alias("total_logged_h"),
        pl.when(pl.col("Target").cast(pl.Float64, strict=False).is_not_null() &
                (pl.col("Target").cast(pl.Float64, strict=False) > 0))
          .then(
              ((pl.col("sum_productive").cast(pl.Float64, strict=False).fill_null(0.0) + pl.col("_idle_sum"))
               / pl.col("Target").cast(pl.Float64, strict=False) * 100).round(1)
          )
          .otherwise(None)
          .alias("total_logged_pct"),
    ]).drop("_idle_sum")

    df = (
        df
        .join(_low_note_ids, on=["Date_Converted", "Email Id"], how="left")
        .with_columns(
            pl.col("_note_flag").fill_null("").alias("note")
        )
        .drop("_note_flag")
    )
    _avail_csv = [c for c in _CSV_DISPLAY_COLS if c in df.columns]
    df = df.select(_avail_csv)
    for _c in [c for c in _DT_COLS if c in df.columns]:
        df = df.with_columns(
            pl.col(_c).dt.strftime("%Y-%m-%d %H:%M:%S").alias(_c)
        )
    _hhmm_h_csv = [c for c in HHMM_COLS_HOURS if c in df.columns]
    _hhmm_in_csv = _hhmm_h_csv + [c for c in HHMM_COLS_MINS if c in df.columns]
    df = df.with_columns(
        [_hours_to_hhmm(c) for c in _hhmm_h_csv] +
        ([_late_to_hhmm("late_vs_sched_min")]  if "late_vs_sched_min"  in df.columns else []) +
        ([_early_to_hhmm("early_leave_min")]   if "early_leave_min"    in df.columns else [])
    )
    df = df.with_columns([
        (pl.lit("'") + pl.col(c)).alias(c) for c in _hhmm_in_csv if c in df.columns
    ])
    _dec_avail = [c for c in _DECIMAL_COLS if c in df.columns]
    df = df.with_columns([
        pl.col(c).cast(pl.Float64, strict=False).round(2).alias(c)
        for c in _dec_avail
        if c not in _hhmm_in_csv
    ])
    return df

_rta_export = (
    RTA_CAST
    .filter(
        (pl.col("Date_Converted") >= pl.lit(start_date).str.strptime(pl.Date, "%Y-%m-%d")) &
        (pl.col("Date_Converted") <= pl.lit(end_date).str.strptime(pl.Date, "%Y-%m-%d"))
    )
)
_rta_export = _format_csv(_rta_export)
_rta_export.write_csv(_out)
print(f"RTA export: {_rta_export.height:,} rows → {_out}")

low_total_logged = (
    low_prod_display
    .filter(
        pl.col("total_logged_pct").is_not_null() &
        (pl.col("total_logged_pct") < TOTAL_LOGGED_THRESHOLD)
    )
    .sort(["Date_Converted", "total_logged_pct"])
)

_hhmm_h_avail = [c for c in HHMM_COLS_HOURS if c in low_total_logged.columns]
low_total_logged = low_total_logged.with_columns(
    [_hours_to_hhmm(c) for c in _hhmm_h_avail] +
    ([_late_to_hhmm("late_vs_sched_min")]  if "late_vs_sched_min"  in low_total_logged.columns else []) +
    ([_early_to_hhmm("early_leave_min")]   if "early_leave_min"    in low_total_logged.columns else [])
)

with pl.Config(tbl_rows=200, tbl_cols=60, fmt_str_lengths=40, tbl_width_chars=1000):
    display(low_total_logged)


RTA export: 356 rows → C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/rta_report_30_days.csv


Date_Converted,Week_Monday,Employee Name,Email Id,IEX ID,OracleID,LOB,LOB_2,Detail Status,Alias,Supervisor Name,First Shift,Fluctuate Shift,Shift Tracking,Datetime_Fluctuate_Start_Shift,Datetime_Fluctuate_End_Shift,Target,start,end,actual_span_h,late_vs_sched_min,early_leave_min,sum_productive,productive_pct,productive_gap_h,duration,break,over_break,exceed_break,break_count,lunch,over_lunch,other_status,training-idle,coaching-idle,total_logged_h,total_logged_pct
date,str,str,str,str,str,str,str,str,str,str,str,str,str,datetime[μs],datetime[μs],f64,datetime[ms],datetime[ms],str,str,str,str,f64,str,str,str,str,f64,f64,str,str,str,str,str,str,f64
2026-09-04,"""2026-08-31""","""NGUYEN HOANG LAM VU""","""hoanglamvu.nguyen@concentrix.com""","""3036686""","""103585973""","""Lodging""","""LG Chat""","""Lodging Production""","""Thomas""","""Tran Hoang My Anh""","""2200-0700""","""2200-0700""","""PR""",2026-09-04 22:00:00,2026-09-05 07:00:00,7.5,2026-09-04 23:29:12,2026-09-05 07:03:57,"""7:34""","""1:29""","""0:00""","""5:20""",71.2,"""2:09""","""7:34""","""0:43""","""0:13""",0.0,0.0,"""1:15""","""0:15""","""0:16""","""0:00""","""0:00""","""5:36""",74.8
2026-09-04,"""2026-08-31""","""HUYNH NGOC HUE TRANG""","""ngochuetrang.huynh@concentrix.com""","""3084756""","""102477510""","""LG Chat CSG""","""LG Chat CSG""","""Lodging Production""","""Jane""","""Nguyen Thi Anh Thu""","""0500-1400""","""0500-1400""","""PR""",2026-09-04 05:00:00,2026-09-04 14:00:00,7.5,2026-09-04 05:28:00,2026-09-04 13:59:10,"""8:31""","""0:28""","""0:00""","""6:38""",88.5,"""0:52""","""8:28""","""0:40""","""0:10""",0.0,0.0,"""1:09""","""0:09""","""0:00""","""0:00""","""0:00""","""6:38""",88.5
2026-09-05,"""2026-08-31""","""NGUYEN HOANG LAM VU""","""hoanglamvu.nguyen@concentrix.com""","""3036686""","""103585973""","""Lodging""","""LG Chat""","""Lodging Production""","""Thomas""","""Tran Hoang My Anh""","""2200-0700""","""2200-0700""","""PR""",2026-09-05 22:00:00,2026-09-06 07:00:00,7.5,2026-09-05 23:23:55,2026-09-06 07:19:41,"""7:55""","""1:23""","""0:00""","""5:18""",70.8,"""2:11""","""7:55""","""1:08""","""0:38""",0.0,0.0,"""1:01""","""0:01""","""0:26""","""0:00""","""0:00""","""5:45""",76.8
